## General Maintenance Notebook for the Pipeline and the database 

In [1]:
import numpy as np 
import pandas as pd 
from scraper import scrape_model 
import os, sys, time
import requests 
import datetime, dateparser


def parse_game_date(date_parse):    
    hour = date_parse[:date_parse.find(':')]
    minute = date_parse[date_parse.find(':')+1:date_parse.find(' ')]
    am_pm = date_parse[date_parse.find(' ')+1:date_parse.find(',')]

    month_day = date_parse[date_parse.find(',')+2:-6]
    month, day = datetime.datetime(month_day.split(' ')[0], '%b').month , month_day.split(' ')[1]
    # year = date_parse[-4:]

    return datetime.date(int(month), int(day))


def parse_game_date_v2(date_parse):
    dow = date_parse[:date_parse.find(',')]
    month = date_parse[date_parse.find(',')+2:date_parse.find(',')+5]
    month_num = datetime.datetime.strptime(month, '%b').month
    dom = date_parse[date_parse.find(month)+4:]
    return (int(month_num), int(dom))


sched_path = 'formed_data/game_schedule_data'
sched_file_dir = os.listdir(sched_path)
urc_sched_files = [sched_path + '/' + i for i in sched_file_dir if 'URC_' in i]
prem_sched_files = [sched_path + '/' + i for i in sched_file_dir if 'Prem_' in i]
sr_sched_files = [sched_path + '/' + i for i in sched_file_dir if 'SR_' in i]
t14_sched_files = [sched_path + '/' + i for i in sched_file_dir if 'T14_' in i]

game_path = 'formed_data/game_data'
game_file_dir = os.listdir(game_path)
urc_game_files = [game_path + '/' + i for i in game_file_dir if 'URC_' in i]
prem_game_files = [game_path + '/' + i for i in game_file_dir if 'Prem_' in i]
sr_game_files = [game_path + '/' + i for i in game_file_dir if 'SR_' in i]
t14_game_files = [game_path + '/' + i for i in game_file_dir if 'T14_' in i]


In [2]:
def assign_season(df):
    if df['month'] >= 9:
        league_season = str((df['season'])) + '-' + str(df['season'] + 1)
    elif df['league_id'] == 242041:
        league_season = df['season']
    else:
        league_season = str(df['season'] - 1) + '-' + str(df['season'])
    return league_season
    

def rejoin_date_df(sched_df, game_df,path):

    sched_df[['month', 'day']] = sched_df['date'].apply(parse_game_date_v2).apply(pd.Series)
    sched_df['date_formatted'] = sched_df.apply(lambda x: datetime.datetime(x['season'], x['month'], x['day']), axis=1)
    sched_df['league_season'] = sched_df.apply(assign_season, axis=1)

    merged_df = game_df.merge(
        sched_df[['game_id', 'date_formatted', 'league_season']],
        how='left',
        on='game_id'
    )

    grouped_df = merged_df.groupby('league_season')
    season_dict = {name: group for name, group in grouped_df}

    for season, value in season_dict.items():
        value.to_csv(path+'_'+str(season)+'.csv', index=False)
        print("DF for season {} saved to {}".format(str(season), path+'_'+str(season)+'.csv'))


def run_joiner_output(sched_path, game_path, output_path):
    sched_concat_df = pd.concat(list(map(pd.read_csv, sched_path)))
    game_concat_df = pd.concat(list(map(pd.read_csv, game_path)))

    rejoin_date_df(
        sched_df=sched_concat_df, 
        game_df=game_concat_df, 
        path=output_path
    )

    

In [3]:
t14_sched = pd.concat(list(map(pd.read_csv, t14_sched_files)))
t14_sched

,Unnamed: 0,date,home_team,away_team,home_team_abbr,away_team_abbr,game_link,score,home_score,away_score,competition,stadium,game_id,league_id,season
0,0,"Sat, Dec 4",Bordeaux Begles,Stade Toulousain,UNI,STA,/rugby/match/_/gameId/594117/league/270559,17 - 7,17,7,French Top 14,"Stade Chaban-Delmas, Bordeaux",594117.0,270559.0,2021
1,1,"Sat, Nov 27",Stade Toulousain,Brive,STA,BRIV,/rugby/match/_/gameId/594114/league/270559,18 - 11,18,11,French Top 14,"Stade Ernest-Wallon, Toulouse",594114.0,270559.0,2021
2,2,"Sat, Nov 6",Stade Toulousain,Perpignan,STA,USA,/rugby/match/_/gameId/594108/league/270559,37 - 15,37,15,French Top 14,"Stade Ernest-Wallon, Toulouse",594108.0,270559.0,2021
3,3,"Sun, Oct 31",Racing 92,Stade Toulousain,RAC,STA,/rugby/match/_/gameId/594098/league/270559,27 - 18,27,18,French Top 14,"Paris La Defense Arena, Nanterre",594098.0,270559.0,2021
4,4,"Sat, Oct 23",Stade Toulousain,Castres Olympique,STA,CAS,/rugby/match/_/gameId/594094/league/270559,41 - 0,41,0,French Top 14,"Stade Ernest-Wallon, Toulouse",594094.0,270559.0,2021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,308,"Sat, Oct 5",Agen,Bayonne,AGN,BAY,NaN,27 - 29,27,29,French Top 14,"Alfred Armandie, Agen",NaN,NaN,2019
184,312,"Sat, Aug 31",Agen,Brive,AGN,BRIV,NaN,16 - 10,16,10,French Top 14,"Alfred Armandie, Agen",NaN,NaN,2019
185,316,"Sat, May 4",Grenoble,Agen,GRENO,AGN,/rugby/match/_/gameId/293211/league/270559,11 - 29,11,29,French Top 14,"Stade des Alpes, Grenoble",293211.0,270559.0,2019
186,323,"Sat, Feb 23",Perpignan,Agen,USA,AGN,/rugby/match/_/gameId/293157/league/270559,13 - 20,13,20,French Top 14,"Stade Aime Giral, Perpignan",293157.0,270559.0,2019


In [4]:
t14_sched[t14_sched['date'] == 'Jue, Abr 29']

,Unnamed: 0,date,home_team,away_team,home_team_abbr,away_team_abbr,game_link,score,home_score,away_score,competition,stadium,game_id,league_id,season
199,343,"Jue, Abr 29",Bayonne,Castres Olympique,BAY,CAS,/rugby/match/_/gameId/593379/league/270559,23 - 26,23,26,French Top 14,"Stade Jean Dauger, Bayonne",593379.0,270559.0,2021


In [5]:
print(dateparser.parse('Jue, Abr 29', languages=['es']))

2025-04-29 00:00:00


In [7]:
dateparser.parse('Jue, Abr 29', languages=['es']).replace(year=2021)

datetime.datetime(2021, 4, 29, 0, 0)

In [12]:
dateparser.parse(t14_sched[t14_sched['date'] == 'Jue, Abr 29']['date'].iloc[0][5:8], languages=['fr'], date_formats='%b')

In [ ]:
t14_file_path = 'formed_data/game_data_v2/T14_game_data_v2'

run_joiner_output(t14_sched_files, t14_game_files, t14_file_path)


ValueError: time data 'Abr' does not match format '%b'

In [33]:
def test_func(date_parse):
    try:
        dow = date_parse[:date_parse.find(',')]
        month = date_parse[date_parse.find(',')+2:date_parse.find(',')+5]
        month_num = datetime.datetime.strptime(month, '%b').month
        # return month_num
    except:
        print(date_parse)
        return 'error'

In [19]:
urc_sched_df = pd.concat(list(map(pd.read_csv, urc_sched_files)))

In [20]:
urc_sched_df

,Unnamed: 0,date,home_team,away_team,home_team_abbr,away_team_abbr,game_link,score,home_score,away_score,competition,stadium,game_id,league_id,season
0,0,"Fri, Dec 3",Leinster,Connacht,LEI,CON,/rugby/match/_/gameId/594385/league/270557,47 - 19,47,19,United Rugby Championship,"RDS Arena, Dublin",594385.0,270557.0,2021
1,1,"Sat, Nov 27",Leinster,Ulster,LEI,ULS,/rugby/match/_/gameId/594381/league/270557,10 - 20,10,20,United Rugby Championship,"RDS Arena, Dublin",594381.0,270557.0,2021
2,2,"Fri, Oct 22",Glasgow Warriors,Leinster,GLA,LEI,/rugby/match/_/gameId/594369/league/270557,15 - 31,15,31,United Rugby Championship,"Scotstoun Stadium, Glasgow",594369.0,270557.0,2021
3,3,"Sat, Oct 16",Leinster,Scarlets,LEI,SCA,/rugby/match/_/gameId/594364/league/270557,50 - 15,50,15,United Rugby Championship,"RDS Arena, Dublin",594364.0,270557.0,2021
4,4,"Sat, Oct 9",Leinster,Zebre,LEI,ZEB,/rugby/match/_/gameId/594353/league/270557,43 - 7,43,7,United Rugby Championship,"RDS Arena, Dublin",594353.0,270557.0,2021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131,214,"Sat, Oct 28",Ospreys,Zebre,OSP,ZEB,/rugby/match/_/gameId/597934/league/270557,34 - 31,34,31,United Rugby Championship,"Swansea.com Stadium, Swansea",597934.0,270557.0,2023
132,218,"Sat, Mar 25",Ospreys,Dragons,OSP,DRA,/rugby/match/_/gameId/599506/league/270557,37 - 18,37,18,United Rugby Championship,"Swansea.com Stadium, Swansea",599506.0,270557.0,2023
133,221,"Sun, Jan 29",Zebre,Ospreys,ZEB,OSP,/rugby/match/_/gameId/599483/league/270557,24 - 28,24,28,United Rugby Championship,"Stadio Sergio Lanfranchi, Parma",599483.0,270557.0,2023
134,232,"Sat, Apr 22",Dragons,Scarlets,DRA,SCA,/rugby/match/_/gameId/599523/league/270557,31 - 14,31,14,United Rugby Championship,"Principality Stadium, Cardiff",599523.0,270557.0,2023


In [27]:
t14_sched_df['date'].isnull().sum()

np.int64(0)

In [34]:
import dateparser

ModuleNotFoundError: No module named 'dateparser'

In [32]:
t14_sched_df

,Unnamed: 0,date,home_team,away_team,home_team_abbr,away_team_abbr,game_link,score,home_score,away_score,competition,stadium,game_id,league_id,season
0,0,"Sat, Dec 4",Bordeaux Begles,Stade Toulousain,UNI,STA,/rugby/match/_/gameId/594117/league/270559,17 - 7,17,7,French Top 14,"Stade Chaban-Delmas, Bordeaux",594117.0,270559.0,2021
1,1,"Sat, Nov 27",Stade Toulousain,Brive,STA,BRIV,/rugby/match/_/gameId/594114/league/270559,18 - 11,18,11,French Top 14,"Stade Ernest-Wallon, Toulouse",594114.0,270559.0,2021
2,2,"Sat, Nov 6",Stade Toulousain,Perpignan,STA,USA,/rugby/match/_/gameId/594108/league/270559,37 - 15,37,15,French Top 14,"Stade Ernest-Wallon, Toulouse",594108.0,270559.0,2021
3,3,"Sun, Oct 31",Racing 92,Stade Toulousain,RAC,STA,/rugby/match/_/gameId/594098/league/270559,27 - 18,27,18,French Top 14,"Paris La Defense Arena, Nanterre",594098.0,270559.0,2021
4,4,"Sat, Oct 23",Stade Toulousain,Castres Olympique,STA,CAS,/rugby/match/_/gameId/594094/league/270559,41 - 0,41,0,French Top 14,"Stade Ernest-Wallon, Toulouse",594094.0,270559.0,2021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,308,"Sat, Oct 5",Agen,Bayonne,AGN,BAY,NaN,27 - 29,27,29,French Top 14,"Alfred Armandie, Agen",NaN,NaN,2019
184,312,"Sat, Aug 31",Agen,Brive,AGN,BRIV,NaN,16 - 10,16,10,French Top 14,"Alfred Armandie, Agen",NaN,NaN,2019
185,316,"Sat, May 4",Grenoble,Agen,GRENO,AGN,/rugby/match/_/gameId/293211/league/270559,11 - 29,11,29,French Top 14,"Stade des Alpes, Grenoble",293211.0,270559.0,2019
186,323,"Sat, Feb 23",Perpignan,Agen,USA,AGN,/rugby/match/_/gameId/293157/league/270559,13 - 20,13,20,French Top 14,"Stade Aime Giral, Perpignan",293157.0,270559.0,2019


In [31]:
for i in range(len(t14_sched_df)):
    test_func(t14_sched_df['date'].iloc[i])
    

Jue, Abr 29
Vie, Abr 16
Vie, Ene 22
Sáb, Ene 16
Sáb, Dic 31
Vie, Dic 23
Sáb, Dic 3
Sáb, Abr 30
Sáb, Abr 23
Sáb, Abr 2
Sáb, Ene 29
Sáb, Ene 8
Sáb, Ene 1


In [13]:
tester = t14_sched_df['date'].iloc[0]
tester

'Sat, Dec 4'

In [21]:
tester[tester.find(',')+2:-6]

''

In [22]:
tester

'Sat, Dec 4'

In [23]:
tester[:tester.find('.')]

'Sat, Dec '

In [24]:
tester[tester.find(',')+2:tester.find(',')+5]

'Dec'

In [25]:
datetime.datetime.strptime(tester[tester.find(',')+2:tester.find(',')+5], '%b').month

12

In [26]:
tester[tester.find(tester[tester.find(',')+2:tester.find(',')+5])+4:]

'4'

In [ ]:
dow = date_parse[:date_parse.find(',')]
month = date_parse[date_parse.find(',')+2:date_parse.find(',')+5]
month_num = datetime.datetime.strptime(month, '%b').month
dom = date_parse[date_parse.find(month)+4:]

In [12]:
t14_sched_df = pd.concat(list(map(pd.read_csv, t14_sched_files)))
t14_sched_df

,Unnamed: 0,date,home_team,away_team,home_team_abbr,away_team_abbr,game_link,score,home_score,away_score,competition,stadium,game_id,league_id,season
0,0,"Sat, Dec 4",Bordeaux Begles,Stade Toulousain,UNI,STA,/rugby/match/_/gameId/594117/league/270559,17 - 7,17,7,French Top 14,"Stade Chaban-Delmas, Bordeaux",594117.0,270559.0,2021
1,1,"Sat, Nov 27",Stade Toulousain,Brive,STA,BRIV,/rugby/match/_/gameId/594114/league/270559,18 - 11,18,11,French Top 14,"Stade Ernest-Wallon, Toulouse",594114.0,270559.0,2021
2,2,"Sat, Nov 6",Stade Toulousain,Perpignan,STA,USA,/rugby/match/_/gameId/594108/league/270559,37 - 15,37,15,French Top 14,"Stade Ernest-Wallon, Toulouse",594108.0,270559.0,2021
3,3,"Sun, Oct 31",Racing 92,Stade Toulousain,RAC,STA,/rugby/match/_/gameId/594098/league/270559,27 - 18,27,18,French Top 14,"Paris La Defense Arena, Nanterre",594098.0,270559.0,2021
4,4,"Sat, Oct 23",Stade Toulousain,Castres Olympique,STA,CAS,/rugby/match/_/gameId/594094/league/270559,41 - 0,41,0,French Top 14,"Stade Ernest-Wallon, Toulouse",594094.0,270559.0,2021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,308,"Sat, Oct 5",Agen,Bayonne,AGN,BAY,NaN,27 - 29,27,29,French Top 14,"Alfred Armandie, Agen",NaN,NaN,2019
184,312,"Sat, Aug 31",Agen,Brive,AGN,BRIV,NaN,16 - 10,16,10,French Top 14,"Alfred Armandie, Agen",NaN,NaN,2019
185,316,"Sat, May 4",Grenoble,Agen,GRENO,AGN,/rugby/match/_/gameId/293211/league/270559,11 - 29,11,29,French Top 14,"Stade des Alpes, Grenoble",293211.0,270559.0,2019
186,323,"Sat, Feb 23",Perpignan,Agen,USA,AGN,/rugby/match/_/gameId/293157/league/270559,13 - 20,13,20,French Top 14,"Stade Aime Giral, Perpignan",293157.0,270559.0,2019


In [10]:
urc_file_path = 'formed_data/game_data_v2/URC_game_data_v2'
prem_file_path = 'formed_data/game_data_v2/Prem_game_data_v2'
sr_file_path = 'formed_data/game_data_v2/SR_game_data_v2'

# run_joiner_output(urc_sched_files, urc_game_files, urc_file_path)
# run_joiner_output(prem_sched_files, prem_game_files, prem_file_path)
run_joiner_output(sr_sched_files, sr_game_files, sr_file_path)


DF for season 2020 saved to formed_data/game_data_v2/SR_game_data_v2_2020.csv
DF for season 2022 saved to formed_data/game_data_v2/SR_game_data_v2_2022.csv
DF for season 2023 saved to formed_data/game_data_v2/SR_game_data_v2_2023.csv
DF for season 2024 saved to formed_data/game_data_v2/SR_game_data_v2_2024.csv


In [4]:
df_rev = pd.read_csv('formed_data/game_data_v2/URC_game_data_v2_2019-2020.csv')
df_rev.head()

,game_id,league_id,home_team,home_team_score,away_team,away_team_score,home_tries,away_tries,home_conversions,away_conversions,...,away_mauls_won_total_num,away_mauls_won_percent,away_scrum_num,away_scrum_total_num,away_scrum_won_percent,away_lineout_num,away_lineout_total_num,away_lineout_won_percent,date_formatted,league_season
0,592889,270557,Ulster,10,Leinster,28,1.0,3.0,1.0,2.0,...,7.0,1.0,6.0,6.0,1.00,14.0,14.0,1.00,2020-08-29,2019-2020
1,592884,270557,Leinster,27,Munster,25,3.0,3.0,3.0,2.0,...,10.0,0.9,4.0,4.0,1.00,13.0,13.0,1.00,2020-08-22,2019-2020
2,294787,270557,Leinster,55,Glasgow Warriors,19,9.0,3.0,5.0,2.0,...,4.0,1.0,8.0,8.0,1.00,5.0,6.0,0.83,2020-02-28,2019-2020
3,294781,270557,Ospreys,13,Leinster,21,1.0,3.0,1.0,3.0,...,5.0,1.0,12.0,12.0,1.00,9.0,13.0,0.69,2020-02-21,2019-2020
4,294775,270557,Leinster,36,Cheetahs,12,5.0,2.0,4.0,1.0,...,4.0,1.0,11.0,12.0,0.91,10.0,11.0,0.90,2020-02-15,2019-2020


In [5]:
df_rev.columns

Index(['game_id', 'league_id', 'home_team', 'home_team_score', 'away_team',
       'away_team_score', 'home_tries', 'away_tries', 'home_conversions',
       'away_conversions', 'home_penalty_goals', 'away_penalty_goals',
       'home_kick_percent', 'away_kick_percent', 'home_total_meters',
       'away_total_meters', 'home_kfh', 'away_kfh', 'home_pass_meters',
       'away_pass_meters', 'home_runs', 'away_runs', 'home_possession_1h_2h',
       'home_territory_1h_2h', 'home_clean_breaks', 'home_defenders_beaten',
       'home_offloads', 'home_rucks_won', 'home_mauls_won',
       'home_turnovers_conceeded', 'away_possession_1h_2h',
       'away_territory_1h_2h', 'away_clean_breaks', 'away_defenders_beaten',
       'away_offloads', 'away_rucks_won', 'away_mauls_won',
       'away_turnovers_conceeded', 'home_total_possession',
       'home_total_territory', 'away_total_possesion', 'away_total_territory',
       'home_scrum', 'home_lineout', 'away_scrum', 'away_lineout',
       'home_tackle

In [8]:
grouped_df = urc_test.groupby('league_season')
df_dict = {name: group for name, group in grouped_df}


In [10]:
for key, value in df_dict.items():
    print(key)

2019-2020
2020-2021
2021-2022
2022-2023
2023-2024


In [9]:
df_dict['2019-2020']

,game_id,league_id,home_team,home_team_score,away_team,away_team_score,home_tries,away_tries,home_conversions,away_conversions,...,away_mauls_won_total_num,away_mauls_won_percent,away_scrum_num,away_scrum_total_num,away_scrum_won_percent,away_lineout_num,away_lineout_total_num,away_lineout_won_percent,date_formatted,league_season
413,592889,270557,Ulster,10,Leinster,28,1.0,3.0,1.0,2.0,...,7.0,1.00,6.0,6.0,1.00,14.0,14.0,1.00,2020-08-29,2019-2020
414,592884,270557,Leinster,27,Munster,25,3.0,3.0,3.0,2.0,...,10.0,0.90,4.0,4.0,1.00,13.0,13.0,1.00,2020-08-22,2019-2020
415,294787,270557,Leinster,55,Glasgow Warriors,19,9.0,3.0,5.0,2.0,...,4.0,1.00,8.0,8.0,1.00,5.0,6.0,0.83,2020-02-28,2019-2020
416,294781,270557,Ospreys,13,Leinster,21,1.0,3.0,1.0,3.0,...,5.0,1.00,12.0,12.0,1.00,9.0,13.0,0.69,2020-02-21,2019-2020
417,294775,270557,Leinster,36,Cheetahs,12,5.0,2.0,4.0,1.0,...,4.0,1.00,11.0,12.0,0.91,10.0,11.0,0.90,2020-02-15,2019-2020
418,294771,270557,Leinster,54,Connacht,7,8.0,1.0,7.0,1.0,...,3.0,1.00,5.0,5.0,1.00,10.0,13.0,0.76,2020-01-04,2019-2020
429,592886,270557,Connacht,26,Ulster,20,4.0,2.0,3.0,2.0,...,3.0,0.66,13.0,13.0,1.00,11.0,15.0,0.73,2020-08-23,2019-2020
430,294784,270557,Ulster,20,Cheetahs,10,2.0,1.0,2.0,1.0,...,6.0,1.00,1.0,1.0,1.00,17.0,20.0,0.85,2020-02-22,2019-2020
431,294777,270557,Ospreys,26,Ulster,24,3.0,3.0,1.0,3.0,...,6.0,0.83,4.0,4.0,1.00,12.0,14.0,0.85,2020-02-15,2019-2020
432,294766,270557,Ulster,38,Munster,17,5.0,2.0,5.0,2.0,...,2.0,1.00,6.0,6.0,1.00,9.0,10.0,0.90,2020-01-03,2019-2020


In [6]:
df_dict.keys()

dict_keys(['2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024'])

In [4]:
urc_test.head()

,game_id,league_id,home_team,home_team_score,away_team,away_team_score,home_tries,away_tries,home_conversions,away_conversions,...,away_mauls_won_total_num,away_mauls_won_percent,away_scrum_num,away_scrum_total_num,away_scrum_won_percent,away_lineout_num,away_lineout_total_num,away_lineout_won_percent,date_formatted,league_season
0,598075,270557,Munster,10,Glasgow Warriors,17,1.0,2.0,1.0,2.0,...,5.0,1.0,11.0,14.0,0.78,9.0,12.0,0.75,2024-06-15,2023-2024
1,598070,270557,Munster,23,Ospreys,7,2.0,1.0,2.0,1.0,...,2.0,1.0,5.0,7.0,0.71,9.0,10.0,0.90,2024-06-07,2023-2024
2,598069,270557,Munster,29,Ulster,24,4.0,3.0,3.0,3.0,...,3.0,1.0,5.0,5.0,1.00,7.0,9.0,0.77,2024-06-01,2023-2024
3,598055,270557,Edinburgh,26,Munster,29,2.0,4.0,2.0,3.0,...,2.0,1.0,4.0,4.0,1.00,8.0,10.0,0.80,2024-05-17,2023-2024
4,598052,270557,Munster,47,Connacht,12,7.0,2.0,6.0,1.0,...,1.0,1.0,6.0,8.0,0.75,7.0,10.0,0.70,2024-05-11,2023-2024


In [ ]:
run_joiner_output(urc_sched_files, urc_game_files, 'game_data_v2')
run_joiner_output(prem_sched_files, prem_game_files, 'game_data_v2')
run_joiner_output(prem_sched_files, prem_game_files, 'game_data_v2')


In [3]:
urc_sched_df = pd.concat(list(map(pd.read_csv, urc_sched_files)))
urc_game_df = pd.concat(list(map(pd.read_csv, urc_game_files)))

prem_sched_df = pd.concat(list(map(pd.read_csv, prem_sched_files)))
prem_game_df = pd.concat(list(map(pd.read_csv, prem_game_files)))

prem_sched_df = pd.concat(list(map(pd.read_csv, prem_sched_files)))
prem_game_df = pd.concat(list(map(pd.read_csv, prem_game_files)))

sr_sched_df = pd.concat(list(map(pd.read_csv, sr_sched_files)))
sr_game_df = pd.concat(list(map(pd.read_csv, sr_game_files)))


In [6]:
urc_game_df.columns

Index(['game_id', 'league_id', 'home_team', 'home_team_score', 'away_team',
       'away_team_score', 'home_tries', 'away_tries', 'home_conversions',
       'away_conversions', 'home_penalty_goals', 'away_penalty_goals',
       'home_kick_percent', 'away_kick_percent', 'home_total_meters',
       'away_total_meters', 'home_kfh', 'away_kfh', 'home_pass_meters',
       'away_pass_meters', 'home_runs', 'away_runs', 'home_possession_1h_2h',
       'home_territory_1h_2h', 'home_clean_breaks', 'home_defenders_beaten',
       'home_offloads', 'home_rucks_won', 'home_mauls_won',
       'home_turnovers_conceeded', 'away_possession_1h_2h',
       'away_territory_1h_2h', 'away_clean_breaks', 'away_defenders_beaten',
       'away_offloads', 'away_rucks_won', 'away_mauls_won',
       'away_turnovers_conceeded', 'home_total_possession',
       'home_total_territory', 'away_total_possesion', 'away_total_territory',
       'home_scrum', 'home_lineout', 'away_scrum', 'away_lineout',
       'home_tackle

In [6]:
urc_sched_df[['month', 'day']] = urc_sched_df['date'].apply(parse_game_date_v2).apply(pd.Series)
urc_sched_df[['date', 'month', 'day']]

urc_sched_df['date_formatted'] = urc_sched_df.apply(lambda x: datetime.datetime(x['season'], x['month'], x['day']), axis=1)
urc_sched_df[['date', 'month', 'day', 'season', 'date_formatted']]

,date,month,day,season,date_formatted
0,"Fri, Dec 3",12,3,2021,2021-12-03
1,"Sat, Nov 27",11,27,2021,2021-11-27
2,"Fri, Oct 22",10,22,2021,2021-10-22
3,"Sat, Oct 16",10,16,2021,2021-10-16
4,"Sat, Oct 9",10,9,2021,2021-10-09
...,...,...,...,...,...
131,"Sat, Oct 28",10,28,2023,2023-10-28
132,"Sat, Mar 25",3,25,2023,2023-03-25
133,"Sun, Jan 29",1,29,2023,2023-01-29
134,"Sat, Apr 22",4,22,2023,2023-04-22


In [7]:
urc_sched_df['league_season'] = urc_sched_df.apply(assign_season, axis=1)
urc_sched_df[['date', 'month', 'day', 'season', 'date_formatted', 'league_season']]

,date,month,day,season,date_formatted,league_season
0,"Fri, Dec 3",12,3,2021,2021-12-03,2021-2022
1,"Sat, Nov 27",11,27,2021,2021-11-27,2021-2022
2,"Fri, Oct 22",10,22,2021,2021-10-22,2021-2022
3,"Sat, Oct 16",10,16,2021,2021-10-16,2021-2022
4,"Sat, Oct 9",10,9,2021,2021-10-09,2021-2022
...,...,...,...,...,...,...
131,"Sat, Oct 28",10,28,2023,2023-10-28,2023-2024
132,"Sat, Mar 25",3,25,2023,2023-03-25,2022-2023
133,"Sun, Jan 29",1,29,2023,2023-01-29,2022-2023
134,"Sat, Apr 22",4,22,2023,2023-04-22,2022-2023
